# Maria's HW2: LLM Paraphrase Encoder Subgroup Analysis
**Изолированный эксперимент.** Не изменяет файлы Анастасии или Дилары. Все артефакты сохраняются в Google Drive.

In [ ]:
# 1. GPU и Google Drive (HF_HOME направлен в Drive ДО загрузки модели)
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import os
from google.colab import drive
drive.mount('/content/drive')

os.environ['HF_HOME'] = '/content/drive/MyDrive/huggingface_cache'
os.makedirs(os.environ['HF_HOME'], exist_ok=True)

import pathlib
WORK = pathlib.Path('/content/drive/MyDrive/far-trans-hw2/maria_outputs')
WORK.mkdir(parents=True, exist_ok=True)
print('Work directory:', WORK)

In [ ]:
# 2. Репозиторий курса (строго зафиксированный коммит)
COURSE_COMMIT = 'f6850861ea20a51e1fb018ee4e5f840c2ac759e0'
!rm -rf /content/RecSys-LLMs && git clone -q https://github.com/dryjins/RecSys-LLMs.git /content/RecSys-LLMs
!git -C /content/RecSys-LLMs checkout -q {COURSE_COMMIT}
COURSE = '/content/RecSys-LLMs/project'

!git -C /content/RecSys-LLMs log -1 --format='Course commit: %H'

In [ ]:
# 3. Командный репозиторий (ветка Марии)
!rm -rf /content/team_repo && git clone -q https://github.com/getmanyan/far-trans-group7.git /content/team_repo
!git -C /content/team_repo fetch origin
!git -C /content/team_repo checkout -q maria/hw2-paraphrase-encoder

TEAM_ROOT = '/content/team_repo'
MARIA_CODE = '/content/team_repo/maria_hw2'
print('Maria code ready at:', MARIA_CODE)
!git -C /content/team_repo log -1 --format='Team commit: %H (%s)'

In [ ]:
# 4. Данные
import zipfile
ZIP = pathlib.Path('/content/drive/MyDrive/far-trans-hw2/FAR-Trans.zip')
if not ZIP.exists():
    from google.colab import files
    print("Please upload FAR-Trans.zip")
    up = files.upload()
    ZIP.write_bytes(up[next(iter(up))])

with zipfile.ZipFile(ZIP) as z:
    z.extractall('/content/data')
DATA = '/content/data/FAR-Trans'
!ls -la {DATA}

In [ ]:
# 5. Зависимости
!pip -q install transformers==4.57.6 huggingface_hub pandas
import torch
print('torch:', torch.__version__, '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)

In [ ]:
# 6. Unit-тесты логики подгрупп Марии
!cd {MARIA_CODE} && python -m unittest test_maria.py -v

In [ ]:
# 7. Загрузка модели (кэшируется в HF_HOME на Google Drive)
from huggingface_hub import snapshot_download
MODEL_ID = 'Qwen/Qwen3-1.7B'
REVISION = '70d244cc86ccca08cf5af4e1e306ecf908b1ad5e'
MODEL_PATH = snapshot_download(MODEL_ID, revision=REVISION)
print('Model path:', MODEL_PATH)

In [ ]:
# 8. Baseline (4 метода курса)
print("Running baselines...")
!cd {COURSE}/far-trans && python starter.py run --data-dir {DATA} --output-dir {WORK}/baseline

import json
baseline_metrics = WORK / 'baseline' / 'metrics.json'
if baseline_metrics.exists():
    with open(baseline_metrics, 'r') as f:
        bm = json.load(f)
    print("✅ Baseline metrics loaded. Methods:", list(bm['methods'].keys()))
else:
    raise FileNotFoundError("❌ Baseline metrics not found.")

In [ ]:
# 9. Эксперимент Марии (передаём MODEL_ID и REVISION, а не локальный путь)
print("Running Maria's experiment via Anastasia's wrapper...")
!cd {MARIA_CODE} && python run_maria_experiment.py \
    --team-root {TEAM_ROOT} \
    --course-dir {COURSE} \
    --data-dir {DATA} \
    --output-dir {WORK}/maria_llmrec \
    --model {MODEL_ID} \
    --revision {REVISION}

In [ ]:
# 10. Проверка совпадения protocol параметров и baseline метрик
import json

baseline_protocol = json.load(open(WORK / 'baseline' / 'protocol.json'))
baseline_metrics = json.load(open(WORK / 'baseline' / 'metrics.json'))

maria_manifest = json.load(open(WORK / 'maria_llmrec' / 'run_manifest.json'))
maria_metrics = json.load(open(WORK / 'maria_llmrec' / 'metrics.json'))

assert maria_manifest['runtime']['model'] == MODEL_ID, f"Model mismatch: {maria_manifest['runtime']['model']} vs {MODEL_ID}"
assert maria_manifest['runtime']['requested_revision'] == REVISION, f"Revision mismatch: {maria_manifest['runtime']['requested_revision']} vs {REVISION}"

protocol_keys = ['users', 'cutoff', 'end_exclusive', 'candidate_count']
for key in protocol_keys:
    assert baseline_protocol[key] == maria_manifest[key], f"Mismatch in {key}: {baseline_protocol[key]} vs {maria_manifest[key]}"

assert baseline_protocol['source_sha256'] == maria_manifest['data_sha256'], f"Data hash mismatch: {baseline_protocol['source_sha256']} vs {maria_manifest['data_sha256']}"

baseline_methods = ['popularity', 'cbf', 'latest_buy', 'latest_buy_or_sell']
for method in baseline_methods:
    bm = baseline_metrics['methods'][method]
    mm = maria_metrics[method]

    assert bm['users'] == mm['users'], f"Users mismatch for {method}"
    assert bm['nonempty_truth_users'] == mm['nonempty_truth_users'], f"Nonempty truth users mismatch for {method}"
    assert bm['ndcg_at_10'] == mm['ndcg_at_10'], f"nDCG@10 mismatch for {method}: {bm['ndcg_at_10']} vs {mm['ndcg_at_10']}"
    assert bm['ndcg_at_10_nonempty'] == mm['ndcg_at_10_nonempty'], f"nDCG@10 nonempty mismatch for {method}: {bm['ndcg_at_10_nonempty']} vs {mm['ndcg_at_10_nonempty']}"

print("✅ All protocol parameters match between baseline and maria_llmrec")
print(f"✅ Model in manifest: {maria_manifest['runtime']['model']}, Revision: {maria_manifest['runtime']['requested_revision']}")
print(f"✅ All 4 baseline metrics match exactly")

In [ ]:
# 11. Анализ подгрупп Марии
print("Running subgroup analysis...")
!cd {MARIA_CODE} && python analyze_maria.py \
    --data-dir {DATA} \
    --course-dir {COURSE} \
    --run-dir {WORK}/maria_llmrec \
    --out {WORK}/results_maria.md

In [ ]:
# 12. Вывод результатов
from IPython.display import Markdown
try:
    display(Markdown(open(f'{WORK}/results_maria.md').read()))
except FileNotFoundError:
    print("❌ Results file not found. Check previous cell output for errors.")

In [ ]:
# 13. Упаковка артефактов (без датасетов и весов модели)
!cd {WORK} && rm -f maria_hw2_outputs.zip && zip -qr maria_hw2_outputs.zip . -x "*.zip" -x "*FAR-Trans*" -x "*huggingface_cache*"
from google.colab import files
files.download(f'{WORK}/maria_hw2_outputs.zip')